# w4 quantization

✔ PyTorch 2.6
✔ transformers 4.51
✔ coremltools 8.0
✔ torchvision 0.21.0
✔ torchaudio 2.6.0

In [2]:
import torch
import coremltools as ct
from transformers import AutoModelForCausalLM, AutoTokenizer

model_id = "Qwen/Qwen3-1.7B" 
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, torch_dtype=torch.float16, attn_implementation="eager", low_cpu_mem_usage=True)

scikit-learn version 1.7.1 is not supported. Minimum required version: 0.17. Maximum required version: 1.5.1. Disabling scikit-learn conversion API.
Torch version 2.6.0 has not been tested with coremltools. You may run into unexpected errors. Torch 2.4.0 is the most recent version that has been tested.
/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 2/2 [00:05<00:00,  2.84s/it]


In [3]:
model = model.to(torch.float16)
model.to("cpu")
model.config.use_cache = False
model.eval()

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (v_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (up_proj): Linear(in_features=2048, out_features=6144, bias=False)
          (down_proj): Linear(in_features=6144, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen3RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwe

In [4]:
for name, b in model.named_buffers():
    if b.dtype != torch.float16:
        print("BUFFER NOT FP16:", name, b.dtype)

# check the dtypes of the model parameters - {torch.float16}
unique_dtypes = set(p.dtype for p in model.parameters())
print(unique_dtypes)

# check rotary embeddings
#model.model.rotary_emb.inv_freq = model.model.rotary_emb.inv_freq.to(torch.float16)
print(model.model.rotary_emb.inv_freq.dtype)

{torch.float16}
torch.float16


In [5]:
# dry run

# torch.Size([1, 1, 151936])

input_ids = torch.zeros((1, 1), dtype=torch.long)
with torch.no_grad():
    outputs = model(input_ids)

print(outputs.logits.shape)

torch.Size([1, 1, 151936])


In [6]:
example_input = torch.zeros((1, 2048), dtype=torch.long)
#model.config._attn_implementation = "eager"


for p in model.parameters():
    assert p.dtype == torch.float16

print(model.config._attn_implementation)

for name, module in model.named_modules():
    if "attn" in name.lower():
        print(name, type(module))


eager
model.layers.0.self_attn <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>
model.layers.0.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.k_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.v_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.o_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.0.self_attn.q_norm <class 'transformers.models.qwen3.modeling_qwen3.Qwen3RMSNorm'>
model.layers.0.self_attn.k_norm <class 'transformers.models.qwen3.modeling_qwen3.Qwen3RMSNorm'>
model.layers.1.self_attn <class 'transformers.models.qwen3.modeling_qwen3.Qwen3Attention'>
model.layers.1.self_attn.q_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.k_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.v_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.o_proj <class 'torch.nn.modules.linear.Linear'>
model.layers.1.self_attn.q_norm <cla

# Convert to CoreML - Exported program

In [11]:
# torch export

MAX_SEQ_LEN = 2048
DEFAULT_SEQ_LEN = 128

# context length
example_input = (torch.zeros((1, DEFAULT_SEQ_LEN), dtype=torch.long),)
#example_input = (torch.zeros((1, 1024), dtype=torch.long),)

from torch.export import Dim

seq_dim = Dim("seq_len", min=1, max=MAX_SEQ_LEN)


from torch.export import export

aten_ops = list(torch.ops.aten.__dict__.values())

exported_program = export(
    model,
    example_input,
    dynamic_shapes={
        "input_ids": {1: torch.export.Dim("seq_len", min=1, max=MAX_SEQ_LEN)}
    },
)

# Run decompositions
exported_program = exported_program.run_decompositions({})


In [12]:
print(exported_program.dialect)
print(type(exported_program))

graph_str = str(exported_program.graph_module.graph)
print(exported_program.dialect)
print("alias present:", "alias" in str(exported_program.graph_module.graph))
print("aten.diff present:", "aten.diff" in graph_str)

ATEN
<class 'torch.export.exported_program.ExportedProgram'>
ATEN
alias present: False
aten.diff present: False


In [13]:
from collections import defaultdict

op_summary = defaultdict(int)
unsupported = []

for node in exported_program.graph_module.graph.nodes:
    target = node.target
    
    # Only inspect callable ops
    if hasattr(target, "__module__"):
        module_name = target.__module__
        op_summary[module_name] += 1
        
        # Flag non-aten namespaces
        if not module_name.startswith("torch._ops.aten"):
            unsupported.append((module_name, target))
    else:
        # Non-callable targets (rare, but include for completeness)
        op_summary[str(type(target))] += 1

print("\n=== Operator Modules Present ===")
for k, v in op_summary.items():
    print(f"{k} -> {v}")

print("\n=== Unsupported Ops Detected ===")
for mod, op in unsupported:
    print(f"{mod} :: {op}")



=== Operator Modules Present ===
<class 'str'> -> 313
torch._ops.aten -> 2870
_operator -> 1

=== Unsupported Ops Detected ===
_operator :: <built-in function add>


In [14]:
import numpy as np
states = []
for layer_idx in range(28):  # num_hidden_layers
    key_state = ct.StateType(
        wrapped_type=ct.TensorType(
            shape=(1, 8, ct.RangeDim(0, 2048), 128),  # Dynamic seq_len
            dtype=np.float16
        ),
        name=f"key_cache_{layer_idx}"
    )
    value_state = ct.StateType(
        wrapped_type=ct.TensorType(
            shape=(1, 8, ct.RangeDim(0, 2048), 128),  # Dynamic seq_len
            dtype=np.float16
        ),
        name=f"value_cache_{layer_idx}"
    )
    states.append(key_state)
    states.append(value_state)

In [ ]:
# Convert to Core ML FP16 - Direct

import coremltools as ct
import numpy as np

mlmodel = ct.convert(
    exported_program,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16,
    minimum_deployment_target=ct.target.iOS17,
    skip_model_load=True,
    package_dir="models/Qwen3_1_7B_fp16_2048_flex_def_128_2.mlpackage",
    debug=True,
    #states=states,    
    #pass_pipeline=ct.PassPipeline.DEFAULT,
    #state_implementation="MLState"
)

#inputs=[
#       ct.TensorType(
#           name="input_ids",
#            shape=ct.Shape(
#                shape=(1, ct.RangeDim(lower_bound=1, upper_bound=MAX_SEQ_LEN, default=DEFAULT_SEQ_LEN))
#            ),
#            dtype=np.int32
#        )
#   ]

Converting PyTorch Frontend ==> MIL Ops:  20%|██        | 584/2871 [03:40<06:43,  5.66 ops/s]  

In [ ]:
from coremltools.optimize.coreml import linear_quantize_weights
import coremltools.optimize as cto


config = cto.coreml.OptimizationConfig(
        global_config=cto.coreml.OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int4")
    )

quantized_model = linear_quantize_weights(
    mlmodel,
    config=config,
    joint_compression=True
)

Running compression pass linear_quantize_weights: 100%|██████████| 315/315 [00:37<00:00,  8.45 ops/s]
Running MIL frontend_milinternal pipeline: 0 passes [00:00, ? passes/s]
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 27.55 passes/s]


In [ ]:
#quantized_model.save("models/Qwen3_1_7B_w4.mlpackage")
quantized_model.save("models/Qwen3_1_7B_w4_2048_flex_def_128_2.mlpackage")

# Export through Trace - Flex context length

In [ ]:
# trace 
import torch._logging
import logging
torch._logging.set_logs(dynamo=logging.DEBUG, graph_code=True)

with torch.no_grad():
    traced_model = torch.jit.trace(
        model,
        example_input,
        check_trace=False,
        strict=False
    )

print(traced_model.graph)

# scripted model
#scripted_model = torch.jit.script(model)

In [ ]:
print("CHECKS")
graph_str = str(traced_model.graph)

print("---- Checking for FP32 in graph ----")

has_fp32 = False

for node in traced_model.graph.nodes():
    for output in node.outputs():
        if "Float" in str(output.type()):
            print("FP32 node:", node)
            has_fp32 = True

if not has_fp32:
    print("✅ Graph is clean FP16")
print("------------------")

checks = {
    "aten::diff present": "aten::diff" in graph_str,
    "prims present": "prims::" in graph_str,
    "alias present": "alias" in graph_str,
    "higher_order present": "higher_order" in graph_str,
    "training dialect present": "training" in graph_str,
}

for k, v in checks.items():
    print(f"{k}: {v}")

from coremltools.converters.mil.frontend.torch.ops import _TORCH_OPS_REGISTRY

graph_str = str(traced_model.graph)

used_ops = set()

for line in graph_str.split("\n"):
    line = line.strip()
    
    if " = aten::" in line:
        op = line.split(" = ")[1].split("(")[0]
        used_ops.add(op.replace("aten::", ""))

print("=== ATEN Ops Used In Model ===")
for op in sorted(used_ops):
    print(op)

print("\nTotal used ATEN ops:", len(used_ops))

supported_ops = set(_TORCH_OPS_REGISTRY.name_to_func_mapping.keys())

print("Number of supported ATEN ops:", len(supported_ops))
unsupported = []

for op in used_ops:
    op_name = op.replace("aten::", "")
    #op_name = op
    if op_name not in supported_ops:
        unsupported.append(op)

print("Unsupported ATEN ops:")
for op in unsupported:
    print(op)

CHECKS
---- Checking for FP32 in graph ----
✅ Graph is clean FP16
------------------
aten::diff present: False
prims present: False
alias present: False
higher_order present: False
training dialect present: False
=== ATEN Ops Used In Model ===
slice

Total used ATEN ops: 1
Number of supported ATEN ops: 378
Unsupported ATEN ops:


In [ ]:
print(traced_model.code)

def forward(self,
    input_ids: Tensor) -> Dict[str, Tensor]:
  lm_head = self.lm_head
  model = self.model
  model0 = self.model
  embed_tokens = model0.embed_tokens
  weight = embed_tokens.weight
  _0 = torch.slice((model).forward(input_ids, ), 0, 0, 9223372036854775807)
  _1 = torch.slice(_0, 1, 0, 9223372036854775807)
  input = torch.slice(_1, 2, 0, 9223372036854775807)
  _2 = {"logits": (lm_head).forward(weight, input, )}
  return _2



In [ ]:
# Through jit trace

import coremltools as ct
import numpy as np

input_shape = ct.EnumeratedShapes(
    shapes=[
        [1, 128],
        [1, 256],
        [1, 512],
        [1, 1024],
        [1, 2048],
    ],
    default=[1, 128],
)

mlmodel = ct.convert(
    traced_model,
    convert_to="mlprogram",
    compute_precision=ct.precision.FLOAT16,
    minimum_deployment_target=ct.target.iOS18,
    skip_model_load=True,
    package_dir="models/Qwen3_1_7B_fp16_2048_flex_def_128.mlpackage",
    inputs=[ct.TensorType(shape=input_shape, dtype=np.int32)],
    debug=True,
)


### Flex context length (NOT TESTED)

In [ ]:
import coremltools as ct
import torch

# Define dynamic dimension
seq_dim = ct.RangeDim(lower_bound=1, upper_bound=2048)

mlmodel = ct.convert(
    exported_program,
    convert_to="mlprogram",
    inputs=[
        ct.TensorType(
            name="input_ids",
            shape=(1, seq_dim),
            dtype=int,
        )
    ],
)

mlmodel.save("Qwen3_1_7B_fp16_2048_flex.mlpackage")

In [ ]:
from coremltools.optimize.coreml import linear_quantize_weights
import coremltools.optimize as cto


config = cto.coreml.OptimizationConfig(
        global_config=cto.coreml.OpLinearQuantizerConfig(mode="linear_symmetric", dtype="int4")
    )

quantized_model = linear_quantize_weights(
    mlmodel,
    config=config,
    joint_compression=True
)

quantized_model.save("models/Qwen3_1_7B_w4_2048_flex.mlpackage")

# Misc code

In [15]:
for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        print("Node:", node)
        print("Inputs:", node.args)

for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        print("Node:", node)
        print("Target:", node.target)
        print("Args:", node.args)
        print("Users:", list(node.users))
        print("-" * 50)

for node in exported_program.graph_module.graph.nodes:
    if "diff" in str(node.target):
        diff_node = node
        break

print("Diff node:", diff_node)
print("Input node:", diff_node.args[0])
producer = diff_node.args[0]
print("Producer:", producer)
print("Producer target:", producer.target)
print("Producer args:", producer.args)

Node: diff
Inputs: (unsqueeze, 1, -1, sub)
Node: diff
Target: aten.diff.default
Args: (unsqueeze, 1, -1, sub)
Users: [ne]
--------------------------------------------------
Diff node: diff
Input node: unsqueeze
Producer: unsqueeze
Producer target: aten.unsqueeze.default
Producer args: (arange, 0)


In [21]:
import transformers.models.qwen3.modeling_qwen3 as modeling_qwen3
import inspect

print(inspect.getsourcefile(modeling_qwen3))

source_lines, start_line = inspect.getsourcelines(modeling_qwen3)

for i, line in enumerate(source_lines):
    if "arange" in line:
        l_no = i
        print(f"{start_line + i}: {line.strip()}")

source_lines, start_line = inspect.getsourcelines(modeling_qwen3)

for j in range(l_no-8, l_no+12):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")

print("---------------------")

for j in range(390, 480):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")


/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/transformers/models/qwen3/modeling_qwen3.py
130: base ** (torch.arange(0, dim, 2, dtype=torch.int64).to(device=device, dtype=torch.float) / dim)
407: cache_position = torch.arange(
399:         if inputs_embeds is None:
400:             inputs_embeds = self.embed_tokens(input_ids)
401: 
402:         if use_cache and past_key_values is None:
403:             past_key_values = DynamicCache(config=self.config)
404: 
405:         if cache_position is None:
406:             past_seen_tokens = past_key_values.get_seq_length() if past_key_values is not None else 0
407:             cache_position = torch.arange(
408:                 past_seen_tokens, past_seen_tokens + inputs_embeds.shape[1], device=inputs_embeds.device
409:             )
410: 
411:         if position_ids is None:
412:             position_ids = cache_position.unsqueeze(0)
413: 
414:         # It may already have been prepared by e.g. `generate`
415:         if not

In [23]:
for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        print(f"{start_line + i}: {line.strip()}")

print("---------------------")

for line in source_lines:
    stripped = line.strip()
    
    # Stop when first class or function appears
    if stripped.startswith("class ") or stripped.startswith("def "):
        break
    
    print(line.rstrip())

---------------------
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
#           This file was automatically generated from src/transformers/models/qwen3/modular_qwen3.py.
#               Do NOT edit this file manually as any edits will be overwritten by the generation of
#             the file from the modular. If any change should be done, please apply the change to the
#                          modular_qwen3.py file directly. One of our CI enforces this.
#                🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨🚨
# coding=utf-8
# Copyright 2025 The Qwen team, Alibaba Group and the HuggingFace Inc. team. All rights reserved.
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     http://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is dist

In [27]:
import transformers.masking_utils as masking_utils
import inspect

print(inspect.getsourcefile(masking_utils))

source_lines, start_line = inspect.getsourcelines(masking_utils)

for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        print(f"{start_line + i}: {line.strip()}")

for i, line in enumerate(source_lines):
    if "def create_causal_mask" in line:
        start_idx = i
        break

for j in range(start_idx, start_idx + 200):
    print(f"{start_line + j}: {source_lines[j].rstrip()}")


print("---------------------")

for i, line in enumerate(source_lines):
    if ".diff(" in line or "torch.diff" in line:
        print(f"{start_line + i}: {line.strip()}")

/Users/naren_work/Code/et_coreml/lib/python3.11/site-packages/transformers/masking_utils.py
769: def create_causal_mask(
769: def create_causal_mask(
770:     config: PreTrainedConfig,
771:     input_embeds: torch.Tensor,
772:     attention_mask: Optional[torch.Tensor],
773:     cache_position: torch.Tensor,
774:     past_key_values: Optional[Cache],
775:     position_ids: Optional[torch.Tensor] = None,
776:     or_mask_function: Optional[Callable] = None,
777:     and_mask_function: Optional[Callable] = None,
778: ) -> Optional[Union[torch.Tensor, BlockMask]]:
779:     """
780:     Create a standard causal mask based on the attention implementation used (stored in the config). If `past_key_values`
781:     has an hybrid cache structure, this function will return the mask corresponding to one of the "full_attention" layers (to align
782:     to what is needed in the `modeling_xxx.py` files).
783: 
784:     Args:
785:         config (`PreTrainedConfig`):
786:             The model confi

In [ ]:
#In transformers -> masking_utils.py

"""
position_diff = torch.cat(
    (
        first_dummy_value,
        position_ids[..., 1:] - position_ids[..., :-1],
    ),
    dim=-1,
)
"""

'\nposition_diff = torch.cat(\n    (\n        first_dummy_value,\n        position_ids[..., 1:] - position_ids[..., :-1],\n    ),\n    dim=-1,\n)\n'